# 🏟️ MetLife Stadium - Ookla Q3 2025 REAL DATA Analysis

This notebook downloads **actual historical Ookla Q3 2025 mobile performance data** and creates an interactive 3D visualization.

**Data Source:** `s3://ookla-open-data/parquet/performance/type=mobile/year=2025/quarter=3/`

## Instructions
1. Click **Runtime → Run all**
2. Wait for data download (~185 MB)
3. Download the generated HTML file

In [ ]:
# Install dependencies
!pip install -q duckdb pandas pyproj pako

In [ ]:
import os
import json
import gzip
import base64
import math
import duckdb
import pandas as pd
from datetime import datetime

# Configuration
MAPBOX_TOKEN = "pk.eyJ1IjoiYXpoYXJ6NHUiLCJhIjoiY2pkaHFtbHAxMGV1cDJxbzI0cjFlcWt4eiJ9.D-0A_N0JhPBfOm-CeFZtMQ"
METLIFE_LAT = 40.813778
METLIFE_LON = -74.074310
SEARCH_RADIUS_KM = 0.8  # 800 meters

PARQUET_URL = "https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile/year=2025/quarter=3/2025-07-01_performance_mobile_tiles.parquet"
LOCAL_FILE = "ookla_q3_2025_mobile.parquet"

print("MetLife Stadium:", METLIFE_LAT, METLIFE_LON)
print("Search radius:", SEARCH_RADIUS_KM * 1000, "meters")

## 1. Download Actual Ookla Q3 2025 Data

In [ ]:
%%time
# Download the actual Ookla parquet file (~185 MB)
if not os.path.exists(LOCAL_FILE):
    print("Downloading Ookla Q3 2025 mobile performance data...")
    print(f"URL: {PARQUET_URL}")
    print("This may take 2-5 minutes depending on connection speed...")
    !wget -q --show-progress -O {LOCAL_FILE} "{PARQUET_URL}"
    
file_size = os.path.getsize(LOCAL_FILE) / (1024*1024)
print(f"\n✅ Downloaded: {LOCAL_FILE} ({file_size:.1f} MB)")

## 2. Load and Explore Data with DuckDB

In [ ]:
# Connect to DuckDB and examine the parquet schema
con = duckdb.connect()

print("Parquet Schema:")
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{LOCAL_FILE}')").fetchdf()
display(schema)

# Get total count
total = con.execute(f"SELECT COUNT(*) FROM read_parquet('{LOCAL_FILE}')").fetchone()[0]
print(f"\nTotal records in Ookla Q3 2025: {total:,}")

In [ ]:
# Sample data to understand structure
print("Sample records:")
sample = con.execute(f"SELECT * FROM read_parquet('{LOCAL_FILE}') LIMIT 5").fetchdf()
display(sample)

## 3. Filter for MetLife Stadium Area

In [ ]:
# Quadkey to coordinates conversion
def quadkey_to_tile(quadkey):
    x = y = 0
    zoom = len(quadkey)
    for i, char in enumerate(quadkey):
        bit = zoom - i - 1
        mask = 1 << bit
        if char == '1': x |= mask
        elif char == '2': y |= mask
        elif char == '3': x |= mask; y |= mask
    return x, y, zoom

def tile_to_bbox(x, y, zoom):
    n = 2 ** zoom
    lon_min = x / n * 360.0 - 180.0
    lon_max = (x + 1) / n * 360.0 - 180.0
    lat_max = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * y / n))))
    lat_min = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * (y + 1) / n))))
    return lon_min, lat_min, lon_max, lat_max

def quadkey_to_center(quadkey):
    x, y, zoom = quadkey_to_tile(quadkey)
    lon_min, lat_min, lon_max, lat_max = tile_to_bbox(x, y, zoom)
    return (lat_min + lat_max) / 2, (lon_min + lon_max) / 2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

print("Helper functions loaded.")

In [ ]:
%%time
# Load all data with quadkeys
print("Loading Ookla data...")
all_data = con.execute(f"""
    SELECT quadkey, avg_d_kbps, avg_u_kbps, avg_lat_ms, tests, devices
    FROM read_parquet('{LOCAL_FILE}')
    WHERE quadkey IS NOT NULL
""").fetchdf()

print(f"Loaded {len(all_data):,} records with quadkeys")

In [ ]:
%%time
# Filter for MetLife Stadium area
print(f"Filtering for points within {SEARCH_RADIUS_KM * 1000}m of MetLife Stadium...")

stadium_data = []
processed = 0

for idx, row in all_data.iterrows():
    processed += 1
    if processed % 500000 == 0:
        print(f"  Processed {processed:,} / {len(all_data):,}...")
    
    try:
        qk = str(row['quadkey'])
        lat, lon = quadkey_to_center(qk)
        dist = haversine(METLIFE_LAT, METLIFE_LON, lat, lon)
        
        if dist <= SEARCH_RADIUS_KM:
            # Determine section based on position
            inside = (40.8115 <= lat <= 40.8160 and -74.0785 <= lon <= -74.0700)
            
            if inside:
                if lat > 40.8147: section = 'Upper North'
                elif lat < 40.8130: section = 'Upper South'
                elif lon > -74.0720: section = 'Upper East'
                elif lon < -74.0765: section = 'Upper West'
                else: section = 'Concourse'
            else:
                section = 'Exterior'
            
            stadium_data.append({
                'quadkey': qk,
                'lat': round(lat, 6),
                'lon': round(lon, 6),
                'download_mbps': round(row['avg_d_kbps'] / 1000, 1) if pd.notna(row['avg_d_kbps']) else 0,
                'upload_mbps': round(row['avg_u_kbps'] / 1000, 1) if pd.notna(row['avg_u_kbps']) else 0,
                'latency_ms': round(row['avg_lat_ms'], 1) if pd.notna(row['avg_lat_ms']) else 0,
                'tests': int(row['tests']) if pd.notna(row['tests']) else 0,
                'devices': int(row['devices']) if pd.notna(row['devices']) else 0,
                'distance_km': round(dist, 3),
                'inside': 1 if inside else 0,
                'section': section,
            })
    except:
        continue

df = pd.DataFrame(stadium_data)
print(f"\n✅ Found {len(df)} tiles within {SEARCH_RADIUS_KM * 1000}m of MetLife Stadium!")

if len(df) > 0:
    display(df.head(10))

In [ ]:
# Statistics
if len(df) > 0:
    print("="*60)
    print("METLIFE STADIUM - OOKLA Q3 2025 ACTUAL DATA STATISTICS")
    print("="*60)
    print(f"Coverage Tiles: {len(df)}")
    print(f"Inside Stadium: {df['inside'].sum()}")
    print(f"Total Tests: {df['tests'].sum():,}")
    print(f"Total Devices: {df['devices'].sum():,}")
    print(f"\nDownload Speed:")
    print(f"  Average: {df['download_mbps'].mean():.1f} Mbps")
    print(f"  Maximum: {df['download_mbps'].max():.1f} Mbps")
    print(f"  Minimum: {df['download_mbps'].min():.1f} Mbps")
    print(f"\nUpload Speed:")
    print(f"  Average: {df['upload_mbps'].mean():.1f} Mbps")
    print(f"  Maximum: {df['upload_mbps'].max():.1f} Mbps")
    print(f"\nLatency:")
    print(f"  Average: {df['latency_ms'].mean():.1f} ms")
    print(f"  Minimum: {df['latency_ms'].min():.1f} ms")
    print("="*60)

## 4. Generate Optimized HTML with Gzip-Compressed REAL Data

In [ ]:
# Prepare data for embedding
if len(df) > 0:
    # Create compact data format: [lon, lat, download, upload, latency, tests, inside, section]
    compact_data = []
    for _, row in df.iterrows():
        compact_data.append([
            row['lon'], row['lat'],
            row['download_mbps'], row['upload_mbps'], row['latency_ms'],
            row['tests'], row['inside'], row['section']
        ])
    
    # Statistics
    stats = {
        'n': len(df),
        'int': int(df['inside'].sum()),
        'dl': round(df['download_mbps'].mean(), 1),
        'dl_max': round(df['download_mbps'].max(), 1),
        'ul': round(df['upload_mbps'].mean(), 1),
        'lat': round(df['latency_ms'].mean(), 1),
        'tst': int(df['tests'].sum()),
        'dev': int(df['devices'].sum()),
    }
    
    # DAS locations (simulated for visualization)
    das_locs = [
        [-74.0742, 40.8154, 'DAS-N1'],
        [-74.0755, 40.8154, 'DAS-N2'],
        [-74.0742, 40.8123, 'DAS-S1'],
        [-74.0755, 40.8123, 'DAS-S2'],
        [-74.0712, 40.8138, 'DAS-E1'],
        [-74.0773, 40.8138, 'DAS-W1'],
        [-74.0742, 40.8138, 'DAS-C1'],
    ]
    
    # All data object
    all_data_obj = {
        'pts': compact_data,
        'das': das_locs,
        'stadium': [[-74.0782, 40.8117], [-74.0782, 40.8159], [-74.0703, 40.8159], [-74.0703, 40.8117]],
        'field': [[-74.0755, 40.8130], [-74.0755, 40.8147], [-74.0730, 40.8147], [-74.0730, 40.8130]],
        'center': [METLIFE_LON, METLIFE_LAT],
        'stats': stats,
        'source': 'Ookla Open Data Q3 2025',
        'generated': datetime.now().isoformat(),
    }
    
    # Convert to JSON and compress
    json_str = json.dumps(all_data_obj, separators=(',', ':'))
    json_size = len(json_str.encode('utf-8'))
    
    compressed = gzip.compress(json_str.encode('utf-8'), compresslevel=9)
    compressed_size = len(compressed)
    
    b64_data = base64.b64encode(compressed).decode('ascii')
    
    print(f"JSON size: {json_size:,} bytes ({json_size/1024:.1f} KB)")
    print(f"Gzip compressed: {compressed_size:,} bytes ({compressed_size/1024:.1f} KB)")
    print(f"Compression ratio: {(1 - compressed_size/json_size)*100:.1f}%")
    print(f"Base64 size: {len(b64_data):,} bytes")

In [ ]:
# Generate the complete HTML
html_template = '''<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>MetLife Stadium - Ookla Q3 2025 REAL DATA</title>
    <script src="https://api.mapbox.com/mapbox-gl-js/v3.0.1/mapbox-gl.js"></script>
    <link href="https://api.mapbox.com/mapbox-gl-js/v3.0.1/mapbox-gl.css" rel="stylesheet"/>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/pako/2.1.0/pako.min.js"></script>
    <style>
        :root{--p:#00D4FF;--s:#00FF88;--bg:#0a0e14;--card:#131920;--txt:#E6EDF3;--dim:#8B949E;}
        *{margin:0;padding:0;box-sizing:border-box;}
        body{font-family:system-ui,sans-serif;background:var(--bg);color:var(--txt);overflow:hidden;}
        #map{position:absolute;inset:0;}
        .pnl{position:absolute;background:rgba(10,14,20,0.95);border:1px solid rgba(0,212,255,0.3);border-radius:16px;padding:20px;backdrop-filter:blur(10px);z-index:1000;}
        .pnl-t{font-family:monospace;font-size:1em;color:var(--p);margin-bottom:15px;padding-bottom:10px;border-bottom:1px solid rgba(0,212,255,0.3);}
        .ctrl{top:20px;left:20px;width:280px;}
        .sts{top:20px;right:20px;width:250px;}
        .leg{bottom:40px;left:20px;}
        .lbtn{display:flex;justify-content:space-between;width:100%;padding:10px 12px;margin:5px 0;background:var(--card);border:1px solid transparent;border-radius:8px;color:var(--txt);font-family:monospace;font-size:0.85em;cursor:pointer;}
        .lbtn:hover{border-color:var(--p);}
        .lbtn.act{border-color:var(--s);color:var(--s);}
        .sr{display:flex;justify-content:space-between;padding:6px 0;border-bottom:1px solid rgba(255,255,255,0.05);font-size:0.9em;}
        .sv{font-family:monospace;font-weight:700;color:var(--s);}
        .lgr{width:180px;height:12px;border-radius:4px;margin:8px 0;}
        .lgl{display:flex;justify-content:space-between;font-size:0.75em;color:var(--dim);}
        .vb{position:absolute;bottom:40px;right:20px;display:flex;flex-direction:column;gap:8px;z-index:1000;}
        .vbtn{padding:10px 16px;background:rgba(10,14,20,0.95);border:1px solid rgba(0,212,255,0.3);border-radius:8px;color:var(--txt);font-size:0.8em;cursor:pointer;}
        .ttl{position:absolute;top:20px;left:50%;transform:translateX(-50%);background:rgba(10,14,20,0.9);border:1px solid rgba(0,212,255,0.3);border-radius:12px;padding:12px 25px;text-align:center;z-index:1000;}
        .ttl h1{font-size:1.2em;color:var(--p);}
        .ttl .sub{font-size:0.7em;color:var(--s);margin-top:4px;}
        .mapboxgl-popup-content{background:var(--card)!important;border:1px solid rgba(0,212,255,0.3)!important;border-radius:12px!important;color:var(--txt)!important;padding:15px!important;}
        #ld{position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);color:var(--p);z-index:2000;}
    </style>
</head>
<body>
    <div id="ld">LOADING REAL OOKLA DATA...</div>
    <div id="map"></div>
    <div class="ttl"><h1>🏟️ METLIFE STADIUM</h1><div class="sub">OOKLA Q3 2025 ACTUAL DATA</div></div>
    <div class="pnl ctrl">
        <div class="pnl-t">📡 LAYER CONTROLS</div>
        <button class="lbtn act" id="b-dl" onclick="sL('dl')">⬇️ Download Speed</button>
        <button class="lbtn" id="b-ul" onclick="sL('ul')">⬆️ Upload Speed</button>
        <button class="lbtn" id="b-lat" onclick="sL('lat')">⏱️ Latency</button>
        <div style="margin-top:12px;padding-top:12px;border-top:1px solid rgba(0,212,255,0.3);font-size:0.85em;">
            <label><input type="checkbox" id="t-das" checked onchange="tDAS()"> DAS Antennas</label><br>
            <label><input type="checkbox" id="t-3d" checked onchange="t3D()"> 3D View</label>
        </div>
    </div>
    <div class="pnl sts">
        <div class="pnl-t">📊 REAL DATA STATS</div>
        <div id="stats"></div>
    </div>
    <div class="pnl leg">
        <div id="lg-t" style="font-size:0.8em;color:var(--p);">DOWNLOAD (Mbps)</div>
        <div class="lgr" id="lg-g" style="background:linear-gradient(90deg,#FF0000,#FFFF00,#00FF00);"></div>
        <div class="lgl"><span id="lg-mn">0</span><span id="lg-md">150</span><span id="lg-mx">300</span></div>
    </div>
    <div class="vb">
        <button class="vbtn" onclick="sV('top')">🔽 Top</button>
        <button class="vbtn" onclick="sV('3d')">🎯 3D</button>
        <button class="vbtn" onclick="sV('orb')">🔄 Orbit</button>
    </div>
    <script>
    const COMPRESSED = "''' + b64_data + '''";
    function decompress(b64){const bin=atob(b64);const bytes=new Uint8Array(bin.length);for(let i=0;i<bin.length;i++)bytes[i]=bin.charCodeAt(i);return JSON.parse(pako.inflate(bytes,{to:'string'}));}
    const LAYERS={dl:{idx:2,title:'DOWNLOAD (Mbps)',min:0,max:300,colors:['#FF0000','#FFFF00','#00FF00'],stops:[0,150,300]},ul:{idx:3,title:'UPLOAD (Mbps)',min:0,max:60,colors:['#FF0000','#FFFF00','#00FF00'],stops:[0,30,60]},lat:{idx:4,title:'LATENCY (ms)',min:10,max:80,colors:['#00FF00','#FFFF00','#FF0000'],stops:[10,45,80]}};
    let D,map,cur='dl',orbI=null;
    (async function(){
        D=decompress(COMPRESSED);
        document.getElementById('stats').innerHTML=`<div class="sr"><span>Tiles</span><span class="sv">${D.stats.n}</span></div><div class="sr"><span>Interior</span><span class="sv">${D.stats.int}</span></div><div class="sr"><span>Avg Download</span><span class="sv">${D.stats.dl} Mbps</span></div><div class="sr"><span>Peak Download</span><span class="sv">${D.stats.dl_max} Mbps</span></div><div class="sr"><span>Avg Upload</span><span class="sv">${D.stats.ul} Mbps</span></div><div class="sr"><span>Avg Latency</span><span class="sv">${D.stats.lat} ms</span></div><div class="sr"><span>Total Tests</span><span class="sv">${D.stats.tst.toLocaleString()}</span></div>`;
        const geo={type:'FeatureCollection',features:D.pts.map(p=>({type:'Feature',geometry:{type:'Point',coordinates:[p[0],p[1]]},properties:{dl:p[2],ul:p[3],lat:p[4],tst:p[5],in:p[6],sec:p[7]}}))};
        const dasG={type:'FeatureCollection',features:D.das.map(d=>({type:'Feature',geometry:{type:'Point',coordinates:[d[0],d[1]]},properties:{id:d[2]}}))};
        const stadG={type:'FeatureCollection',features:[{type:'Feature',geometry:{type:'Polygon',coordinates:[[...D.stadium,D.stadium[0]]]}}]};
        const fldG={type:'FeatureCollection',features:[{type:'Feature',geometry:{type:'Polygon',coordinates:[[...D.field,D.field[0]]]}}]};
        mapboxgl.accessToken="''' + MAPBOX_TOKEN + '''";
        map=new mapboxgl.Map({container:'map',style:'mapbox://styles/mapbox/satellite-streets-v12',center:D.center,zoom:16,pitch:60,bearing:-20});
        map.on('load',()=>{
            document.getElementById('ld').style.display='none';
            map.addSource('dem',{type:'raster-dem',url:'mapbox://mapbox.mapbox-terrain-dem-v1',tileSize:512});
            map.setTerrain({source:'dem',exaggeration:1.5});
            map.addLayer({id:'sky',type:'sky',paint:{'sky-type':'atmosphere'}});
            map.addSource('stad',{type:'geojson',data:stadG});map.addLayer({id:'stad-ln',type:'line',source:'stad',paint:{'line-color':'#00D4FF','line-width':3}});
            map.addSource('fld',{type:'geojson',data:fldG});map.addLayer({id:'fld-fl',type:'fill',source:'fld',paint:{'fill-color':'#00FF88','fill-opacity':0.3}});
            map.addSource('das',{type:'geojson',data:dasG});map.addLayer({id:'das-pt',type:'circle',source:'das',paint:{'circle-radius':8,'circle-color':'#FFD700','circle-stroke-width':2,'circle-stroke-color':'#FFF'}});map.addLayer({id:'das-lb',type:'symbol',source:'das',layout:{'text-field':['get','id'],'text-size':10,'text-offset':[0,1.5]},paint:{'text-color':'#FFD700','text-halo-color':'#000','text-halo-width':1}});
            map.addSource('data',{type:'geojson',data:geo});map.addLayer({id:'data-pt',type:'circle',source:'data',paint:{'circle-radius':['interpolate',['linear'],['zoom'],14,5,18,15],'circle-color':['interpolate',['linear'],['get','dl'],0,'#FF0000',150,'#FFFF00',300,'#00FF00'],'circle-opacity':0.85,'circle-stroke-width':1,'circle-stroke-color':'#FFF'}});
            map.on('click','data-pt',e=>{const p=e.features[0].properties;new mapboxgl.Popup().setLngLat(e.lngLat).setHTML(`<div style="font-weight:bold;color:#00D4FF;margin-bottom:8px;">📍 ${p.sec}</div><div style="display:flex;justify-content:space-between;padding:4px 0;"><span>⬇️ Download</span><span style="color:#00FF88;font-weight:bold;">${p.dl} Mbps</span></div><div style="display:flex;justify-content:space-between;padding:4px 0;"><span>⬆️ Upload</span><span style="color:#00FF88;font-weight:bold;">${p.ul} Mbps</span></div><div style="display:flex;justify-content:space-between;padding:4px 0;"><span>⏱️ Latency</span><span style="color:#00FF88;font-weight:bold;">${p.lat} ms</span></div><div style="display:flex;justify-content:space-between;padding:4px 0;"><span>📊 Tests</span><span style="color:#00FF88;font-weight:bold;">${p.tst.toLocaleString()}</span></div>`).addTo(map);});
            map.on('mouseenter','data-pt',()=>map.getCanvas().style.cursor='pointer');map.on('mouseleave','data-pt',()=>map.getCanvas().style.cursor='');
        });
    })();
    function sL(l){cur=l;const L=LAYERS[l];const prop=l;map.setPaintProperty('data-pt','circle-color',['interpolate',['linear'],['get',prop],L.stops[0],L.colors[0],L.stops[1],L.colors[1],L.stops[2],L.colors[2]]);document.getElementById('lg-t').textContent=L.title;document.getElementById('lg-g').style.background=`linear-gradient(90deg,${L.colors.join(',')})`;document.getElementById('lg-mn').textContent=L.min;document.getElementById('lg-md').textContent=Math.round((L.min+L.max)/2);document.getElementById('lg-mx').textContent=L.max;document.querySelectorAll('.lbtn').forEach(b=>b.classList.remove('act'));document.getElementById('b-'+l).classList.add('act');}
    function tDAS(){const v=document.getElementById('t-das').checked?'visible':'none';map.setLayoutProperty('das-pt','visibility',v);map.setLayoutProperty('das-lb','visibility',v);}
    function t3D(){map.easeTo({pitch:document.getElementById('t-3d').checked?60:0,duration:1000});}
    function sV(v){if(orbI){clearInterval(orbI);orbI=null;}if(v==='top')map.easeTo({center:D.center,zoom:16,pitch:0,bearing:0,duration:1500});else if(v==='3d')map.easeTo({center:D.center,zoom:16,pitch:60,bearing:-20,duration:1500});else if(v==='orb'){let b=map.getBearing();orbI=setInterval(()=>{b+=0.5;map.easeTo({bearing:b,duration:50});},50);}}
    </script>
</body>
</html>'''

# Save HTML file
output_file = 'metlife_stadium_ookla_q3_2025_REAL_DATA.html'
with open(output_file, 'w', encoding='utf-8') as f:
    f.write(html_template)

html_size = os.path.getsize(output_file)
print(f"\n✅ Generated: {output_file}")
print(f"   File size: {html_size:,} bytes ({html_size/1024:.1f} KB)")

## 5. Download the HTML File

In [ ]:
# Download the generated HTML file
from google.colab import files
files.download(output_file)
print("\n🎉 Download started! Open the HTML file in your browser.")

In [ ]:
# Also save the filtered data as CSV
csv_file = 'metlife_stadium_ookla_q3_2025_data.csv'
df.to_csv(csv_file, index=False)
print(f"Saved data to: {csv_file}")
files.download(csv_file)